In [ ]:
# Setup: clone the repo and switch to the `cmkd` branch (all CMKD code lives there, not on main).
# Kaggle settings: Accelerator = GPU T4 x2, Internet = ON (needed for git clone and DeiT-Base weights).
!git clone https://github.com/TotallyNotMinh/Environmental-Sound-Recognition-Under-Unseen-Conditions.git
%cd /kaggle/working/Environmental-Sound-Recognition-Under-Unseen-Conditions
!git fetch origin
!git checkout cmkd
!git pull origin cmkd

In [ ]:
!pip install -q h5py scipy opencv-python tqdm

In [ ]:
# Quick check (~3 min, no dataset or teacher needed): KD loss + KD training on synthetic data.
!python losses/distillation.py
# One 16 GB T4 cannot hold 12 AST-Base clips at once: total batch 12 = 4 accum steps x 3 clips.
!python scripts/train_kd.py --mock --num-epoch 2 --num-workers 0 --grad-accum-steps 4 --checkpoint-dir /kaggle/working/smoke/kd/

In [ ]:
# Find the inputs:
#   1) TEACHER (required): "Add Input" -> the output of the CNN notebook (train.ipynb) that contains
#      checkpoints/fsd50k_cnn_b0/best_cnn.pth.
#   2) PREVIOUS KD RUN (optional): to CONTINUE an unfinished KD run, also "Add Input" -> this notebook's
#      previous version output. Its checkpoint is copied here and training resumes from the last finished epoch.
import glob, os, shutil

def find_in_inputs(pattern):
    # Skip the FSD50K dataset folder so the search stays fast.
    return sorted(p for d in glob.glob("/kaggle/input/*") if not d.endswith("datasets")
                  for p in glob.glob(f"{d}/**/{pattern}", recursive=True))

teachers = find_in_inputs("fsd50k_cnn_b0/best_cnn.pth")
if not teachers:
    raise FileNotFoundError("Teacher not found. Add the CNN notebook's output (with checkpoints/fsd50k_cnn_b0/best_cnn.pth) as an input.")
TEACHER = teachers[0]
print(f"Teacher: {TEACHER}")

RUN = "/kaggle/working/checkpoints/fsd50k_kd_ast_base"
os.makedirs(RUN, exist_ok=True)
prev = find_in_inputs("fsd50k_kd_ast_base/checkpoint.pth")
if prev and not os.path.exists(f"{RUN}/checkpoint.pth"):
    src = os.path.dirname(prev[0])
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), RUN)
    print(f"Resuming KD from previous run: {src}")
elif os.path.exists(f"{RUN}/checkpoint.pth"):
    print("Checkpoint already in /kaggle/working -- resuming.")
else:
    print("No previous KD checkpoint -- starting fresh.")

In [ ]:
# Train: distill the frozen CNN teacher into an AST-Base student.
# CMKD FSD50K AST recipe: LR 5e-5, batch 12, 50 epochs, BCE + KD (lambda 0.5, tau 1.0).
# --batch-size is the EFFECTIVE total batch: 12 = 2 GPUs x 2 accum steps x 3 clips per GPU.
# AST has no BatchNorm, so accumulation is identical to a real batch of 12 (it only saves memory).
#
# Stopped after 11 h (Kaggle kills sessions at 12 h). checkpoint.pth is saved after every epoch, so at
# most the epoch in progress is lost. The evaluation cell below still runs and Kaggle saves the outputs.
!timeout --signal=INT --kill-after=5m 11h torchrun --nproc_per_node=2 scripts/train_kd.py \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --teacher-checkpoint {TEACHER} \
        --batch-size 12 \
        --grad-accum-steps 2 \
        --lr 5e-5 \
        --num-epoch 50 \
        --checkpoint-dir {RUN}/ \
        --checkpoint-path {RUN}/checkpoint.pth

In [ ]:
# KD-AST performance: score the best checkpoint on the FSD50K EVAL split (the split the CMKD paper reports;
# paper target for EfficientNet-B0 -> AST-Base is ~0.617 mAP). Saved to {RUN}/eval_eval_best_kd_ast.json.
import pandas as pd

hist = pd.read_csv(f"{RUN}/metrics.csv")
last = int(hist["epoch"].max())
print(f"Epochs finished: {last}/50" + ("" if last >= 50 else "  <-- unfinished: add this version's output as input and re-run to continue"))
print(hist.tail(5).to_string(index=False))

!python scripts/evaluate.py \
        --checkpoint {RUN}/best_kd_ast.pth \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --split eval